# Model 3: Advanced Feature Extraction & XGBoost Meta-Ranker

Two-stage stacked ensemble for MCQ option ranking. A Logistic Regression base model generates calibrated probabilities from TF-IDF features; an XGBoost meta-ranker combines those probabilities with rich similarity and structural features to produce the final top-3 prediction per question.

## 1. Imports & Configuration

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import xgboost as xgb
import wandb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold, train_test_split

print("Imports OK")

In [ ]:
# Kaggle competition paths (hardcoded for cloud execution)
TRAIN_PATH      = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH       = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

# Local fallback for smoke-testing on a laptop
for _var, _alts in [
    ("TRAIN_PATH",      ["data/train.csv",             "../../data/train.csv"]),
    ("TEST_PATH",       ["data/test.csv",              "../../data/test.csv"]),
    ("SAMPLE_SUB_PATH", ["data/sample_submission.csv", "../../data/sample_submission.csv"]),
]:
    if not os.path.exists(globals()[_var]):
        for _p in _alts:
            if os.path.exists(_p):
                globals()[_var] = _p
                break

print(f"TRAIN      → {TRAIN_PATH}")
print(f"TEST       → {TEST_PATH}")

In [ ]:
# W&B — inject key so Kaggle kernels authenticate without interactive login
os.environ["WANDB_API_KEY"] = "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"

wandb.init(
    project="DL-GenAI-Project",
    name="Model_3_XGBoost_Ensemble",
    config={
        "base_model":           "LogisticRegression",
        "meta_model":           "XGBClassifier",
        "word_tfidf_features":  8000,
        "word_tfidf_ngram":     "(1,3)",
        "char_tfidf_features":  4000,
        "char_tfidf_ngram":     "(3,5)",
        "oof_folds":            5,
        "xgb_n_estimators":     300,
        "xgb_lr":               0.05,
        "xgb_max_depth":        5,
        "val_split":            0.15,
    }
)
CFG = wandb.config
print("W&B run:", wandb.run.name)

In [ ]:
# Load data
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

for df in [train_df, test_df]:
    if "context" not in df.columns:
        df["context"] = ""
    df["context"] = df["context"].fillna("")

print("Train:", train_df.shape, "| Test:", test_df.shape)
train_df.head(3)

## 2. Feature Extraction Pipeline

In [ ]:
OPTION_COLS = ["A", "B", "C", "D", "E"]

def clean(text: str) -> str:
    """Lowercase and collapse whitespace. Keeps punctuation for n-gram char features."""
    return re.sub(r"\s+", " ", str(text).lower().strip())

# Build combined query: context + prompt
train_queries = (train_df["context"] + " " + train_df["prompt"]).map(clean).tolist()
test_queries  = (test_df["context"]  + " " + test_df["prompt"]).map(clean).tolist()

# Gather all text to fit a shared vocabulary (prevents OOV on test options)
all_options = []
for df in [train_df, test_df]:
    for col in OPTION_COLS:
        all_options += df[col].fillna("").map(clean).tolist()

full_corpus = train_queries + test_queries + all_options

# ── Vectorizer 1: Word-level TF-IDF (1–3 ngrams) ──────────────────────────
# Captures single words, common two-word phrases, and three-word sequences.
# sublinear_tf dampens very frequent terms via log(1+tf).
word_tfidf = TfidfVectorizer(
    max_features=CFG.word_tfidf_features,
    ngram_range=(1, 3),
    stop_words="english",
    sublinear_tf=True,
    strip_accents="unicode",
)
word_tfidf.fit(full_corpus)
print(f"Word TF-IDF vocab : {len(word_tfidf.vocabulary_)} terms")

# ── Vectorizer 2: Character-level TF-IDF (3–5 grams) ─────────────────────
# Character n-grams handle morphological variants, typos, and short options
# that fool word-level models (e.g., '42' vs '43').
char_tfidf = TfidfVectorizer(
    max_features=CFG.char_tfidf_features,
    ngram_range=(3, 5),
    analyzer="char_wb",    # char_wb pads word boundaries — reduces noise
    sublinear_tf=True,
)
char_tfidf.fit(full_corpus)
print(f"Char TF-IDF vocab : {len(char_tfidf.vocabulary_)} terms")

In [ ]:
def jaccard(set_a: set, set_b: set) -> float:
    """Jaccard index = |A ∩ B| / |A ∪ B|. Returns 0 if both sets are empty."""
    if not set_a and not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


def word_intersection_ratio(query_words: set, opt_words: set) -> float:
    """Fraction of option words that also appear in the query.
    High ratio → the option is topically close to the question."""
    if not opt_words:
        return 0.0
    return len(query_words & opt_words) / len(opt_words)


def extract_features(df: pd.DataFrame, queries: list, is_test: bool = False) -> pd.DataFrame:
    """
    Build a long-format DataFrame: one row per (question, option) pair.

    Feature groups produced per row:
      word_cos_sim  — cosine similarity using word TF-IDF vectors
      char_cos_sim  — cosine similarity using character TF-IDF vectors
      jaccard_sim   — Jaccard index between query and option token sets
      word_intersect— fraction of option words shared with the query
      opt_len       — character length of the option string
      prompt_len    — character length of the prompt string
      len_ratio     — option_len / (prompt_len + 1)  [+1 avoids div-by-zero]
      opt_wc        — word count of the option
      prompt_wc     — word count of the prompt
      wc_ratio      — opt_wc / (prompt_wc + 1)
    """
    # Transform entire query list at once — much faster than inside the loop
    word_q_vecs = word_tfidf.transform(queries)   # sparse (n, word_vocab)
    char_q_vecs = char_tfidf.transform(queries)   # sparse (n, char_vocab)

    records = []
    for i, (_, row) in enumerate(df.iterrows()):
        prompt_txt   = clean(str(row.get("prompt", "")))
        query_tokens = set(prompt_txt.split())
        p_len        = len(prompt_txt)
        p_wc         = len(prompt_txt.split())

        wq_vec = word_q_vecs[i]   # sparse slice for this question
        cq_vec = char_q_vecs[i]

        for opt in OPTION_COLS:
            opt_txt    = clean(str(row.get(opt, "")))
            opt_tokens = set(opt_txt.split())

            # ── Similarity features ──────────────────────────────────────
            wo_vec      = word_tfidf.transform([opt_txt])   # sparse (1, word_vocab)
            co_vec      = char_tfidf.transform([opt_txt])   # sparse (1, char_vocab)
            word_cos    = float(cosine_similarity(wq_vec, wo_vec)[0][0])
            char_cos    = float(cosine_similarity(cq_vec, co_vec)[0][0])
            jac         = jaccard(query_tokens, opt_tokens)
            intersect   = word_intersection_ratio(query_tokens, opt_tokens)

            # ── Structural / length features ─────────────────────────────
            o_len  = len(opt_txt)
            o_wc   = len(opt_txt.split())

            # Binary label — 1 = this option is the correct answer
            label = 0
            if not is_test and "answer" in row:
                label = 1 if str(row["answer"]).strip().upper() == opt else 0

            records.append({
                "id":            row["id"],
                "option_key":    opt,
                "word_cos_sim":  word_cos,
                "char_cos_sim":  char_cos,
                "jaccard_sim":   jac,
                "word_intersect":intersect,
                "opt_len":       o_len,
                "prompt_len":    p_len,
                "len_ratio":     o_len / (p_len + 1),
                "opt_wc":        o_wc,
                "prompt_wc":     p_wc,
                "wc_ratio":      o_wc / (p_wc + 1),
                "label":         label,
            })

    return pd.DataFrame(records)


print("Extracting train features...")
train_feat = extract_features(train_df, train_queries, is_test=False)
print("Extracting test  features...")
test_feat  = extract_features(test_df,  test_queries,  is_test=True)

print(f"Train long-format : {train_feat.shape}  positives={train_feat['label'].sum()}")
print(f"Test  long-format : {test_feat.shape}")
train_feat.head(10)

## 3. Dataset Assembly

In [ ]:
# Feature columns fed to both the base model and the meta-ranker
FEAT_COLS = [
    "word_cos_sim", "char_cos_sim", "jaccard_sim", "word_intersect",
    "opt_len", "prompt_len", "len_ratio", "opt_wc", "prompt_wc", "wc_ratio",
]

X_all  = train_feat[FEAT_COLS].values.astype(np.float32)
y_all  = train_feat["label"].values
X_test = test_feat[FEAT_COLS].values.astype(np.float32)

print(f"Feature matrix — train: {X_all.shape}  test: {X_test.shape}")
print(f"Label balance  — 0: {(y_all==0).sum()}  1: {(y_all==1).sum()}")

## 4. XGBoost Model Training

In [ ]:
# ── Stage 1: Out-of-Fold Logistic Regression ───────────────────────────────
# We generate OOF probabilities from LogReg to use as a stacked feature.
# OOF ensures XGBoost never sees base predictions computed on its own training rows,
# which would introduce data leakage and cause severe overfitting.

skf = StratifiedKFold(n_splits=CFG.oof_folds, shuffle=True, random_state=42)
oof_lr_proba = np.zeros(len(X_all), dtype=np.float32)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_all, y_all)):
    lr = LogisticRegression(
        C=1.0,
        max_iter=300,
        class_weight="balanced",   # corrects 1:4 label imbalance
        solver="lbfgs",
        random_state=42,
    )
    lr.fit(X_all[tr_idx], y_all[tr_idx])
    oof_lr_proba[val_idx] = lr.predict_proba(X_all[val_idx])[:, 1]
    fold_acc = accuracy_score(y_all[val_idx], (oof_lr_proba[val_idx] >= 0.5).astype(int))
    print(f"Fold {fold+1}/{CFG.oof_folds}  LR acc: {fold_acc:.4f}")

oof_lr_acc = accuracy_score(y_all, (oof_lr_proba >= 0.5).astype(int))
oof_lr_ll  = log_loss(y_all, oof_lr_proba)
print(f"\nOOF LR  accuracy: {oof_lr_acc:.4f}  logloss: {oof_lr_ll:.4f}")
wandb.log({"oof_lr_accuracy": oof_lr_acc, "oof_lr_logloss": oof_lr_ll})

# Train final LR on all data to score the test set
lr_final = LogisticRegression(C=1.0, max_iter=300, class_weight="balanced",
                               solver="lbfgs", random_state=42)
lr_final.fit(X_all, y_all)
test_lr_proba = lr_final.predict_proba(X_test)[:, 1]
print(f"Test LR proba mean: {test_lr_proba.mean():.4f}")

In [ ]:
# ── Stage 2: XGBoost Meta-Ranker ───────────────────────────────────────────
# Concatenate LogReg's OOF probability alongside the hand-crafted features.
# XGBoost learns which combination of these signals reliably identifies the
# correct answer, correcting systematic blind spots in the linear base model.

X_meta_train = np.hstack([oof_lr_proba.reshape(-1, 1), X_all])
X_meta_test  = np.hstack([test_lr_proba.reshape(-1, 1), X_test])

X_tr, X_val, y_tr, y_val = train_test_split(
    X_meta_train, y_all,
    test_size=CFG.val_split,
    stratify=y_all,
    random_state=42,
)
print(f"XGBoost train: {X_tr.shape}  val: {X_val.shape}")

# scale_pos_weight = neg / pos — corrects class imbalance for XGBoost's loss
spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
print(f"scale_pos_weight: {spw:.2f}")

meta_clf = xgb.XGBClassifier(
    objective          = "binary:logistic",
    eval_metric        = "logloss",
    n_estimators       = CFG.xgb_n_estimators,   # 300 trees
    learning_rate      = CFG.xgb_lr,             # 0.05
    max_depth          = CFG.xgb_max_depth,       # 5
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    scale_pos_weight   = spw,
    use_label_encoder  = False,
    random_state       = 42,
    early_stopping_rounds = 25,
    verbosity          = 0,
)

meta_clf.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=50,
)
print(f"Best iteration: {meta_clf.best_iteration}")

## 5. Evaluation & Submission Generation

In [ ]:
# Validation metrics
val_proba = meta_clf.predict_proba(X_val)[:, 1]
val_preds = (val_proba >= 0.5).astype(int)

val_acc = accuracy_score(y_val, val_preds)
val_f1  = f1_score(y_val, val_preds, average="macro", zero_division=0)
val_ll  = log_loss(y_val, val_proba)

print(f"XGB val accuracy : {val_acc:.4f}")
print(f"XGB val macro-F1 : {val_f1:.4f}")
print(f"XGB val logloss  : {val_ll:.4f}")

wandb.log({
    "val_accuracy": val_acc,
    "val_f1":       val_f1,
    "val_logloss":  val_ll,
    "best_iter":    meta_clf.best_iteration,
})

In [ ]:
# MAP@3 on the val split
_, val_idx = train_test_split(
    np.arange(len(train_feat)), test_size=CFG.val_split,
    stratify=y_all, random_state=42,
)
val_feat = train_feat.iloc[val_idx].copy()
val_feat["pred_proba"] = val_proba

def ap_at_3(group: pd.DataFrame) -> float:
    """Average Precision@3 for one question group."""
    ranked = group.sort_values("pred_proba", ascending=False).reset_index(drop=True)
    score, hits = 0.0, 0
    for rank, row in ranked.head(3).iterrows():
        if row["label"] == 1:
            hits += 1
            score += hits / (rank + 1)
    return score

map3 = val_feat.groupby("id").apply(ap_at_3).mean()
print(f"Val MAP@3: {map3:.4f}")
wandb.log({"val_map_at_3": map3})

In [ ]:
# Inference on test set → top-3 per question → submission.csv
test_feat["pred_proba"] = meta_clf.predict_proba(X_meta_test)[:, 1]

def top3(group: pd.DataFrame) -> str:
    return " ".join(
        group.sort_values("pred_proba", ascending=False)["option_key"].head(3).tolist()
    )

submission = (
    test_feat
    .groupby("id", sort=False)
    .apply(top3)
    .reset_index()
    .rename(columns={0: "prediction"})
)
submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv — {len(submission)} rows")
print(submission.head(10))

wandb.finish()